# Context Groundedness Scorer — Single-Prompt Demo + RAGTruth Eval

This notebook demonstrates the new [`ContextGroundednessScorer`](../uqlm/scorers/longform/context_groundedness.py) API in `mode="single_prompt"` and then evaluates it on a small, readable RAGTruth sample using [`RAGTruthGrader`](../uqlm/longform/benchmark/ragtruth_grader.py).

## What this notebook does
1. Creates the scorer in single-prompt mode.
2. Runs one compact demo example and prints the generated prompt.
3. Builds a small balanced RAGTruth subset with a simple helper.
4. Runs the scorer on that subset.
5. Computes claim-level and response-level metrics.


## 0. Prerequisites

Make sure Ollama is running and the model is available, for example:

```bash
ollama pull qwen3:8b
```

In [2]:
import os
import json
import ast
from typing import Any, Dict, List

import numpy as np
import pandas as pd
from datasets import load_dataset
from langchain_ollama import ChatOllama

from uqlm import ContextGroundednessScorer
from uqlm.longform.benchmark import RAGTruthGrader
from uqlm.utils.prompts.groundedness_prompts import get_unified_groundedness_prompt

os.environ.setdefault("OLLAMA_HOST", "http://127.0.0.1:11450")

'127.0.0.1:11450'

## 1. Initialize the local LLM

In [10]:
MODEL_NAME = "qwen3:14b"

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0.0,
)

resp = llm.invoke("Say hello in one word.")
print("Smoke test response:", resp.content)

Smoke test response: Hello.


## 2. Create the scorer in single-prompt mode

In [11]:
scorer = ContextGroundednessScorer(
    mode="single_prompt",
    llm=llm,
    aggregation="mean",
    include_reasoning=True,
    include_relevant_context=True,
)

print("Scorer initialized successfully")

Scorer initialized successfully


## 3. Minimal demo example

This example is adapted from RAGTruth dataset

In [12]:
demo_query = "weather in big bear city ca"
demo_context = """0am:The Big Bear City, CA forecast for Mar 06 is 28 degrees and Cloudy. There is 93 percentage chance of rain and 18 mph winds from the West-Northwest. 3am:The Big Bear City, CA forecast for Mar 06 is 28 degrees and Clear. There is 83 percentage chance of rain and 17 mph winds from the West-Northwest. 6am:The Big Bear City, CA forecast for Mar 06 is 25 degrees and Clear. There is 63 percentage chance of rain and 14 mph winds from the West-Northwest. 0am:The Big Bear City, CA forecast for Apr 16 is 48 degrees and Cloudy. There is 33 percentage chance of rain and 4 mph winds from the West-Southwest. 3am:The Big Bear City, CA forecast for Apr 16 is 43 degrees and Overcast. There is 33 percentage chance of rain and 4 mph winds from the West-Southwest. 6am:The Big Bear City, CA forecast for Apr 16 is 43 degrees and Clear. There is 34 percentage chance of rain and 3 mph winds from the West-Southwest. 9am:The Big Bear City, CA forecast for Apr 16 is 50 degrees and Partly cloudy. There is 24 percentage chance of rain and 3 mph winds from the Southwest. 12am:The Big Bear City, CA forecast for Apr 16 is 59 degrees and Sunny. There is 12 percentage chance of rain and 4 mph winds from the South-Southwest."""
demo_answer = """The weather in Big Bear City, CA varies depending on the time of the year. On March 6, the temperature ranges from 25 to 28 degrees with a chance of rain varying from 63% to 93% and winds from the West-Northwest at 14 to 18 mph. On April 16, the temperature ranges from 43 to 59 degrees with a chance of rain between 24% and 34% and winds from the West-Southwest and South-Southwest at 3 to 4 mph."""

In [13]:
demo_prompt = get_unified_groundedness_prompt(
    context=demo_context,
    answer=demo_answer,
    include_reasoning=True,
    include_relevant_context=True,
)

print(demo_prompt[:8000])
if len(demo_prompt) > 8000:
    print("\n... [prompt truncated for display] ...")

## Context (source of truth)
0am:The Big Bear City, CA forecast for Mar 06 is 28 degrees and Cloudy. There is 93 percentage chance of rain and 18 mph winds from the West-Northwest. 3am:The Big Bear City, CA forecast for Mar 06 is 28 degrees and Clear. There is 83 percentage chance of rain and 17 mph winds from the West-Northwest. 6am:The Big Bear City, CA forecast for Mar 06 is 25 degrees and Clear. There is 63 percentage chance of rain and 14 mph winds from the West-Northwest. 0am:The Big Bear City, CA forecast for Apr 16 is 48 degrees and Cloudy. There is 33 percentage chance of rain and 4 mph winds from the West-Southwest. 3am:The Big Bear City, CA forecast for Apr 16 is 43 degrees and Overcast. There is 33 percentage chance of rain and 4 mph winds from the West-Southwest. 6am:The Big Bear City, CA forecast for Apr 16 is 43 degrees and Clear. There is 34 percentage chance of rain and 3 mph winds from the West-Southwest. 9am:The Big Bear City, CA forecast for Apr 16 is 50 degrees and

In [14]:
demo_result = await scorer.score(
    queries=[demo_query],
    contexts=[demo_context],
    answers=[demo_answer],
    return_prompts=True,
    show_progress_bars=False,
)

print("Response score:", demo_result.data["response_scores"][0])
print("Number of extracted claims:", len(demo_result.data["claims_data"][0]))

Response score: 1.0
Number of extracted claims: 7


In [15]:
for i, claim in enumerate(demo_result.data["claims_data"][0], start=1):
    print(f"[{i}] verdict={claim['verdict']:<12} score={claim['score']:.2f}")
    print(" claim:       ", claim["claim"])
    print(" anchor_text: ", claim["anchor_text"])
    print(" offsets:     ", (claim["start_offset"], claim["end_offset"]))
    print(" reasoning:   ", claim["reasoning"][:200])
    print(" relevant_ctx:", claim["relevant_context"][:2])
    print()

[1] verdict=supported    score=1.00
 claim:        The weather in Big Bear City, CA varies depending on the time of the year.
 anchor_text:  The weather in Big Bear City, CA varies depending on the time of the year.
 offsets:      (0, 74)
 reasoning:    The context provides forecasts for two different dates (March 6 and April 16) with distinct weather conditions, supporting the claim that weather varies by time of year.
 relevant_ctx: ['The Big Bear City, CA forecast for Mar 06 is 28 degrees and Cloudy. There is 93 percentage chance of rain and 18 mph winds from the West-Northwest.', 'The Big Bear City, CA forecast for Apr 16 is 48 degrees and Cloudy. There is 33 percentage chance of rain and 4 mph winds from the West-Southwest.']

[2] verdict=supported    score=1.00
 claim:        On March 6, the temperature ranges from 25 to 28 degrees.
 anchor_text:  On March 6, the temperature ranges from 25 to 28 degrees
 offsets:      (75, 131)
 reasoning:    The context lists temperatures of 25°

## 4. Build a small readable RAGTruth subset

Instead of using the full preparation pipeline from [`scripts/data/prepare_ragtruth_test.py`](../../scripts/data/prepare_ragtruth_test.py), we use a compact helper that samples `N` clean and `N` hallucinated examples per task type.

In [16]:
def parse_labels(x: Any) -> List[Dict[str, Any]]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    if isinstance(x, str):
        s = x.strip()
        if not s or s.lower() == "nan":
            return []
        for parser in (json.loads, ast.literal_eval):
            try:
                value = parser(s)
                if isinstance(value, list):
                    return value
                if isinstance(value, dict):
                    return [value]
            except Exception:
                pass
    return []

def add_hallucination_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["answer"] = df["output"]

    def get_processed_counts(x):
        if isinstance(x, dict):
            return int(x.get("evident_conflict", 0) or 0), int(x.get("baseless_info", 0) or 0)
        return 0, 0

    counts = df["hallucination_labels_processed"].apply(get_processed_counts)
    df["evident_conflict"] = counts.apply(lambda t: t[0])
    df["baseless_info"] = counts.apply(lambda t: t[1])
    df["has_hallucination"] = (df["evident_conflict"] > 0) | (df["baseless_info"] > 0)
    return df

def sample_ragtruth_subset(df: pd.DataFrame, n_per_class: int = 5, seed: int = 42) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    sampled_parts = []

    for task_type in sorted(df["task_type"].dropna().unique()):
        task_df = df[df["task_type"] == task_type]
        clean_df = task_df[~task_df["has_hallucination"]]
        hall_df = task_df[task_df["has_hallucination"]]

        n_clean = min(n_per_class, len(clean_df))
        n_hall = min(n_per_class, len(hall_df))

        if n_clean > 0:
            sampled_parts.append(clean_df.sample(n=n_clean, random_state=int(rng.randint(0, 1_000_000))))
        if n_hall > 0:
            sampled_parts.append(hall_df.sample(n=n_hall, random_state=int(rng.randint(0, 1_000_000))))

    sampled_df = pd.concat(sampled_parts, ignore_index=True)
    return sampled_df

def print_subset_stats(df: pd.DataFrame) -> None:
    print(f"Total sampled examples: {len(df)}")
    print()
    print(df.groupby(["task_type", "has_hallucination"]).size().unstack(fill_value=0))
    print()
    print("Overall hallucination rate:", round(df["has_hallucination"].mean(), 3))

In [17]:
raw_ds = load_dataset("wandb/RAGTruth-processed")
ragtruth_df = add_hallucination_flags(raw_ds["train"].to_pandas())
sample_df = sample_ragtruth_subset(ragtruth_df, n_per_class=5, seed=42)
print_subset_stats(sample_df)

Repo card metadata block was not found. Setting CardData to empty.


Total sampled examples: 30

has_hallucination  False  True 
task_type                      
Data2txt               5      5
QA                     5      5
Summary                5      5

Overall hallucination rate: 0.5


## 5. Run the scorer on the sampled subset

In [ ]:
queries = sample_df["query"].tolist()
contexts = sample_df["context"].tolist()
answers = sample_df["answer"].tolist()

eval_result = await scorer.score(
    queries=queries,
    contexts=contexts,
    answers=answers,
    show_progress_bars=True,
)

print(f"Scored {len(eval_result.data['claims_data'])} answers")
print(f"Total extracted claims: {sum(len(x) for x in eval_result.data['claims_data'])}")

Output()

JSON parse error in single_prompt groundedness response: Expecting ',' delimiter: line 40 column 39 (char 2273)

JSON parse error in single_prompt groundedness response: Expecting ',' delimiter: line 19 column 35 (char 980)

JSON parse error in single_prompt groundedness response: Expecting ',' delimiter: line 47 column 38 (char 3334)

JSON parse error in single_prompt groundedness response: Expecting ',' delimiter: line 5 column 42 (char 228)

## 6. Inspect one sampled prediction

In [ ]:
example_idx = int(sample_df[sample_df['has_hallucination']].index[0]) if sample_df['has_hallucination'].any() else 0
row = sample_df.iloc[example_idx]
claims = eval_result.data["claims_data"][example_idx]

print("Task type:", row["task_type"])
print("Has hallucination (GT):", bool(row["has_hallucination"]))
print("Response score:", eval_result.data["response_scores"][example_idx])
print()
print("Answer preview:")
print(row["answer"][:700])
print()
for claim in claims[:8]:
    print(f"- {claim['verdict']:<12} score={claim['score']:.2f} span=({claim['start_offset']}, {claim['end_offset']})")
    print(f"  claim: {claim['claim'][:180]}")
    print(f"  anchor: {claim['anchor_text'][:180]}")
    print()

## 7. Evaluate with RAGTruthGrader

In [ ]:
grader = RAGTruthGrader()
gt_labels = [parse_labels(x) for x in sample_df["hallucination_labels"].tolist()]

graded = grader.evaluate(
    gt_hallucination_labels=gt_labels,
    claim_verdicts=eval_result.data["claims_data"],
    response_scores=eval_result.data["response_scores"],
)

print("Evaluation complete")

In [ ]:
claim_metrics = graded["claim_level"]
response_metrics = graded["response_level"]

metrics_df = pd.DataFrame([
    {
        "claim_recall": claim_metrics["recall"],
        "claim_precision": claim_metrics["precision"],
        "claim_f1": claim_metrics["f1"],
        "n_gt_spans": claim_metrics["n_gt_spans"],
        "n_pred_hallucinations": claim_metrics["n_pred_hallucinations"],
        "n_gt_matched": claim_metrics["n_gt_matched"],
        "n_pred_matched": claim_metrics["n_pred_matched"],
        "response_auc_roc": response_metrics["auc_roc"],
        "n_responses": response_metrics["n_responses"],
        "n_hallucinated": response_metrics["n_hallucinated"],
        "n_clean": response_metrics["n_clean"],
    }
])
metrics_df.round(4)

In [ ]:
confusion = graded["confusion_matrix"]
print(json.dumps(confusion, indent=2, ensure_ascii=False))